# Octahedron constellation — station-keeping interval analysis

This notebook presents `analysis-SK-duration(2).py` as numbered steps, with an explanation before every code cell. It compares exported maneuver costs and position deviations for different station-keeping (SK) intervals.

Run the notebook from top to bottom. The sequence is: define and run the Δv consistency analysis; define plotting and deviation functions; generate the original figures. Function-definition cells do not calculate results until a later cell calls them.

## Code preservation and requirements

The only executable changes are **seven replacements of `Path(__file__).resolve().parent` with `Path.cwd()`**, including occurrences inside functions. Jupyter does not normally define `__file__`. The original algorithms, comments, exclusions, plotting options, repeated imports and execution order are retained. Script cell markers become notebook boundaries, and long sections are split between complete top-level functions.

- Use a Python Jupyter kernel with `pandas`, `numpy` and `matplotlib`. The preserved Tk backend requires Tk support and a desktop display.
- Place this notebook in the project directory containing `data-export`, and ensure that directory is the kernel's working directory. A Jupyter kernel's working directory is not guaranteed to match the notebook's folder in every setup.
- This notebook reads exported CSVs. It does not connect to STK, propagate spacecraft, rerun targeting or generate the source reports.
- Keep a consistent simulation duration, coordinate frame, vector direction and naming scheme across the cases being compared. These properties are not verified by this analysis script.

## Input and output files

| Location relative to the project directory | Contents |
| --- | --- |
| `data-export/maneuver-summaries/Octahedron_*.csv` | Maneuver summaries with metadata and a `--- BEGIN DATA ---` marker. The table needs `Delta V (m/sec)`; the metadata needs `Total Delta V (m/sec)`. |
| `data-export/positions/Octahedron_*.csv` | Position CSVs with the same data-start marker and table columns `x (km)`, `y (km)`, `z (km)`. |
| `data-export/deviations/*_deviations.csv` | Generated tables containing original data and added deviation columns, without the metadata preamble. |
| `results/*.png` | Generated figures, saved at 600 dpi before display. |

Example case name: `Octahedron_SK7d_Xp1000m.csv`. The filename encodes the SK interval and the intended offset (+1000 m along X). These scripts expect the project's processed CSV format; an arbitrary direct STK CSV export may not contain the required metadata marker.

Existing deviation CSVs and figures with identical output names are overwritten on rerun. Source position and maneuver-summary CSVs are read without modification. The original execution cells can create **up to 14 figures** when all requested cases are available: one Δv comparison, one four-panel maximum-deviation figure, one magnitude comparison, nine time-series figures, and two trade-off figures.


## 1. Select the plotting backend

This cell preserves `%matplotlib tk`, so plots appear in separate desktop windows with interactive navigation. Run it before importing `matplotlib.pyplot`. Figures are also saved automatically by the plotting functions.

If Tk is unavailable, or you prefer embedded figures, you can replace this cell with `%matplotlib inline`. That optional change has not been made to the supplied code.

In [1]:
%matplotlib tk


## 2. Import libraries and configure paths and figure sizes

This cell imports the analysis libraries, sets the project directory using `Path.cwd()`, and points `summaries_dir` to `data-export/maneuver-summaries`.

The scale factor sets three figure sizes: 11.25 × 5.25 inches for single plots, 11.25 × 9 inches for four-panel comparisons, and 11.25 × 6 inches for time series. No CSVs are read in this cell.

In [2]:
import pandas as pd
from pathlib import Path
from collections import defaultdict
import re
import numpy as np
import matplotlib.pyplot as plt

# Get the path to the maneuver summaries directory
current_dir = Path.cwd()
summaries_dir = current_dir / "data-export" / "maneuver-summaries"

# Global plot dimensions configuration
scaler = 0.75
PLOT_DIM_SINGLE = (15 * scaler, 7 * scaler)      # Single plot (Delta V vs SK)
PLOT_DIM_4SUBPLOT = (15 * scaler, 12 * scaler)   # 2x2 subplot grid (Max deviations)
PLOT_DIM_TIMESERIES = (15 * scaler, 8 * scaler) # 4x1 subplot grid (Deviations over time)


## 3. Define the SK interval parser

`parse_sk_interval` converts an interval name into a numeric value in hours while keeping its original label. For example, `SK7d` becomes 168 hours and `SK12h` becomes 12 hours. Decimal notation such as `SK0p5h` is also accepted.

Using hours internally lets later functions sort day and hour cases numerically rather than alphabetically. A string that does not match the expected pattern raises `ValueError`. This cell only defines the helper.

In [4]:
def parse_sk_interval(sk_string):
    """
    Parse SK interval string and return numeric value and unit.
    
    Handles formats:
    - SK1d, SK7d, SK14d, etc. (days)
    - SK12h, SK6h, etc. (hours)
    
    Parameters:
    -----------
    sk_string : str
        SK interval string (e.g., "SK7d", "SK12h")
    
    Returns:
    --------
    tuple : (numeric_value_in_hours, original_string)
        numeric_value_in_hours: float value for sorting (hours)
        original_string: str for display
    """
    match = re.match(r'SK(\d+(?:[\.p]\d+)?)(d|h)', sk_string)
    if not match:
        raise ValueError(f"Invalid SK format: {sk_string}")
    
    value = float(match.group(1).replace('p', '.'))
    unit = match.group(2)
    
    # Convert to hours for consistent sorting
    if unit == 'd':
        value_in_hours = value * 24
    else:  # 'h'
        value_in_hours = value
    
    return value_in_hours, sk_string


## 4. Define the maneuver-table Δv calculation

`calculate_delta_v_from_data` locates `--- BEGIN DATA ---`, reads the table beneath it with Pandas, sums `Delta V (m/sec)`, and returns that sum together with the number of table rows.

The official [STK Maneuver Summary reference](https://help.agi.com/stk/SubSystems/dataProviders/Content/html/dataProviders/Maneuver_Summary.htm) defines per-maneuver Δv and distinguishes the integrated value for finite maneuvers. Here the script simply adds the exported scalar values. It does not add velocity-change vectors, select only SK maneuvers, or normalise the sum by elapsed time or number of orbits. The row count represents a maneuver count only if each row is one maneuver.

In [5]:
def calculate_delta_v_from_data(csv_file_path):
    """
    Calculate Delta V by summing all maneuver Delta V values from the data section.
    
    Parameters:
    -----------
    csv_file_path : Path
        Path to the CSV file
    
    Returns:
    --------
    delta_v : float
        Sum of all Delta V values
    num_maneuvers : int
        Number of maneuvers
    """
    # Read the file and find where the actual data starts
    with open(csv_file_path, 'r') as f:
        lines = f.readlines()
    
    # Find the line with actual data header (the one after "--- BEGIN DATA ---")
    data_start_idx = None
    for i, line in enumerate(lines):
        if '--- BEGIN DATA ---' in line:
            data_start_idx = i + 1
            break
    
    if data_start_idx is None:
        raise ValueError("Could not find '--- BEGIN DATA ---' marker in file")
    
    # Read CSV starting from the actual data header
    df = pd.read_csv(csv_file_path, skiprows=data_start_idx)
    
    # Sum the Delta V column
    delta_v_sum = df['Delta V (m/sec)'].sum()
    num_maneuvers = len(df)
    
    return delta_v_sum, num_maneuvers


## 5. Define the metadata Δv reader

`extract_delta_v_from_header` looks for a line containing `Total Delta V (m/sec)` and converts its second comma-separated field to a float. It returns `None` if that field cannot be read or the entry is absent.

This reads a total already stored in the project's CSV metadata. It does not recompute the total and does not call STK. The previous helper provides the independent table sum used for comparison.

In [6]:
def extract_delta_v_from_header(csv_file_path):
    """
    Extract Total Delta V from the CSV header statistics section.
    """
    with open(csv_file_path, 'r') as f:
        lines = f.readlines()
    
    for line in lines:
        if 'Total Delta V (m/sec)' in line:
            parts = line.strip().split(',')
            if len(parts) >= 2:
                try:
                    return float(parts[1])
                except ValueError:
                    return None
    return None


## 6. Define the constellation Δv consistency analysis

`analyze_octahedron_constellations` groups all matching maneuver-summary CSVs by numeric SK interval, then compares each satellite's metadata total against the sum of its maneuver-table values. A difference below `1e-6` m/s passes the comparison.

The function prints satellite details, constellation totals and a final table. It does not save that summary table. Its totals include table sums only for files whose header total was successfully extracted; reading failures are reported and skipped. The reported satellite count is the number of grouped files, not a check that all six directions are present.

This is a check of CSV consistency. Passing it does not establish targeting convergence, physical accuracy, or that a case's maneuver schedule is valid. The analysis is unfiltered by the exclusion lists defined later.

In [7]:
def analyze_octahedron_constellations():
    """
    Analyze all Octahedron constellation CSV files and compare calculated vs header Delta V.
    Supports both SK{days}d and SK{hours}h formats.
    """
    # Group files by constellation (SK interval)
    constellations = defaultdict(list)
    sk_display = {}  # Store display format for each SK interval
    
    # Find all Octahedron CSV files
    csv_files = list(summaries_dir.glob("Octahedron_*.csv"))
    
    if not csv_files:
        print(f"No Octahedron CSV files found in {summaries_dir}")
        return
    
    # Group by SK interval (extract SK{X}d or SK{X}h after "Octahedron_")
    for csv_file in csv_files:
        filename = csv_file.stem
        # Extract SK interval: "Octahedron_SK7d_Xm_1000m" -> "SK7d"
        parts = filename.split('_')
        if len(parts) < 2:
            continue
        
        sk_string = parts[1]  # e.g., "SK7d" or "SK12h"
        
        try:
            sk_hours, sk_display_str = parse_sk_interval(sk_string)
            constellations[sk_hours].append(csv_file)
            sk_display[sk_hours] = sk_display_str
        except ValueError as e:
            print(f"Warning: {e} in file {filename}, skipping...")
            continue
    
    # Sort SK intervals numerically (by hours)
    sk_hours_list = sorted(constellations.keys())
    
    print(f'\n{"="*100}')
    print(f'OCTAHEDRON CONSTELLATION DELTA V ANALYSIS')
    print(f'{"="*100}\n')
    
    summary_data = []
    
    for sk_hours in sk_hours_list:
        csv_files = sorted(constellations[sk_hours])
        sk_string = sk_display[sk_hours]
        
        print(f'--- SK Interval: {sk_string} ---')
        print(f'Number of satellites: {len(csv_files)}\n')
        
        total_delta_v_calculated = 0.0
        total_delta_v_header = 0.0
        constellation_maneuvers = 0
        validation_passed = True
        
        # Process each satellite
        for csv_file in csv_files:
            sat_name = csv_file.stem
            
            # Extract values from header
            delta_v_header = extract_delta_v_from_header(csv_file)
            
            # Calculate from data
            try:
                delta_v_calc, num_maneuvers = calculate_delta_v_from_data(csv_file)
                constellation_maneuvers += num_maneuvers
            except Exception as e:
                print(f'  ✗ {sat_name}: Error reading data - {str(e)}')
                validation_passed = False
                continue
            
            # Compare
            if delta_v_header is not None:
                diff = abs(delta_v_calc - delta_v_header)
                match = '✓' if diff < 1e-6 else '✗'
                
                print(f'  {match} {sat_name}')
                print(f'      Header: {delta_v_header:.10f} m/sec')
                print(f'      Calced: {delta_v_calc:.10f} m/sec')
                
                if diff >= 1e-6:
                    print(f'      Diff:   {diff:.10e} m/sec')
                    validation_passed = False
                
                total_delta_v_header += delta_v_header
                total_delta_v_calculated += delta_v_calc
            else:
                print(f'  ✗ {sat_name}: Could not extract header value')
                validation_passed = False
        
        # Print constellation summary
        print(f'\n  Constellation Total ({sk_string}):')
        print(f'    Total Calculated Delta V: {total_delta_v_calculated:.10f} m/sec')
        print(f'    Total Header Delta V:     {total_delta_v_header:.10f} m/sec')
        print(f'    Total Maneuvers:          {constellation_maneuvers}')
        print(f'    Validation:               {"PASSED ✓" if validation_passed else "FAILED ✗"}')
        
        summary_data.append({
            'SK Interval': sk_string,
            'Num Satellites': len(csv_files),
            'Total Maneuvers': constellation_maneuvers,
            'Delta V (m/sec)': total_delta_v_calculated,
            'Validation': 'PASSED' if validation_passed else 'FAILED'
        })
        
        print()
    
    # Print final summary table
    print(f'{"="*100}')
    print(f'SUMMARY TABLE')
    print(f'{"="*100}\n')
    
    summary_df = pd.DataFrame(summary_data)
    print(summary_df.to_string(index=False))
    
    print(f'\n{"="*100}')
    print(f'Analysis complete!')


## 7. Run the Δv consistency analysis

This cell calls the analysis function and prints the results. The source's `if __name__ == '__main__':` guard is retained; it normally evaluates to true in a Jupyter kernel.

If the maneuver-summary directory is empty or unavailable, the function prints that no matching files were found and returns. Check any failed comparisons before using the plotted totals. Nothing is saved by this cell.

In [8]:
if __name__ == '__main__':
    analyze_octahedron_constellations()



OCTAHEDRON CONSTELLATION DELTA V ANALYSIS

--- SK Interval: SK28d ---
Number of satellites: 6

  ✓ Octahedron_SK28d_Xm1000m
      Header: 0.0045569206 m/sec
      Calced: 0.0045569206 m/sec
  ✓ Octahedron_SK28d_Xp1000m
      Header: 0.0045547978 m/sec
      Calced: 0.0045547979 m/sec
  ✓ Octahedron_SK28d_Ym1000m
      Header: 0.0022904432 m/sec
      Calced: 0.0022904433 m/sec
  ✓ Octahedron_SK28d_Yp1000m
      Header: 0.0022914463 m/sec
      Calced: 0.0022914463 m/sec
  ✓ Octahedron_SK28d_Zm1000m
      Header: 0.0028640885 m/sec
      Calced: 0.0028640885 m/sec
  ✓ Octahedron_SK28d_Zp1000m
      Header: 0.0028639528 m/sec
      Calced: 0.0028639528 m/sec

  Constellation Total (SK28d):
    Total Calculated Delta V: 0.0194216494 m/sec
    Total Header Delta V:     0.0194216492 m/sec
    Total Maneuvers:          42
    Validation:               PASSED ✓

SUMMARY TABLE

SK Interval  Num Satellites  Total Maneuvers  Delta V (m/sec) Validation
      SK28d               6               4

## 8. Load the plotting section setup

The original script repeats its imports and reassigns the maneuver-summary directory before defining the plotting functions. These statements are retained in the same order.

The second script-directory expression is replaced with `Path.cwd()`. This cell still depends on the global figure-size settings defined earlier and performs no plotting itself.

In [9]:
import pandas as pd
from pathlib import Path
from collections import defaultdict
import re
import matplotlib.pyplot as plt
import numpy as np

# Get the path to the maneuver summaries directory
current_dir = Path.cwd()
summaries_dir = current_dir / "data-export" / "maneuver-summaries"


## 9. Define the satellite-name decoder

`extract_satellite_info` reads the SK interval and direction from a filename such as `Octahedron_SK7d_Xm1000m`. It returns the interval, axis (`X`, `Y` or `Z`) and sign (`plus` or `minus`). Unrecognised names return three `None` values.

Later plotting functions group by axis and sign. They do not distinguish different formation sizes within one direction, so use a consistently selected size in these input directories when interpreting a curve as a single size study.

In [10]:
def extract_satellite_info(filename):
    """
    Extract SK interval and satellite direction from filename.
    
    Example: "Octahedron_SK7d_Xm1000m" -> ("SK7d", "X", "minus")
    """
    parts = filename.split('_')
    if len(parts) < 3:
        return None, None, None
    
    sk_string = parts[1]  # e.g., "SK7d"
    direction_part = parts[2]  # e.g., "Xm1000m"
    
    # Extract axis (X, Y, Z) and sign (p for plus, m for minus)
    match = re.match(r'([XYZ])([pm])', direction_part)
    if not match:
        return None, None, None
    
    axis = match.group(1)
    sign = 'plus' if match.group(2) == 'p' else 'minus'
    
    return sk_string, axis, sign


## 10. Define the Δv versus SK interval plot

`plot_delta_v_vs_sk_duration` reads Δv totals from CSV metadata, groups them by the six axis/sign combinations, sorts by SK interval, and applies case-insensitive filename-substring exclusions. X, Y and Z use blue, green and red; plus/minus use light/dark shades.

The x-axis is always logarithmic. `x_axis_days` changes its numeric coordinates from hours to days while custom tick labels retain the original hour/day suffixes. `y_log` controls the y-axis. Values are multiplied by 1000 to convert m/s to mm/s.

**Preserved interpretation issue:** the label says `mm/s/orbit`, but this function does not divide by the number of orbits. That label is justified only if the input totals already represent one orbit. The function saves `results/delta_v_vs_sk_duration...png`, with suffixes for options and actual exclusions, then displays the plot.

In [11]:
def plot_delta_v_vs_sk_duration(exclude_names=None, y_log=True, x_axis_days=False):
    """
    Plot total Delta V vs SK duration for all satellites.
    Uses 3 colors for X, Y, Z axes and light/dark shades for plus/minus.
    
    Parameters:
    -----------
    exclude_names : list, optional
        List of satellite name patterns to exclude from the plot.
        Matches against satellite filenames using substring matching (case-insensitive).
        Examples:
            ['SK3h']              - Exclude all satellites with 'SK3h' in filename
            ['Xm']                - Exclude all Xm satellites
            ['SK1h', 'SK3h']      - Exclude SK1h or SK3h satellites
        If None, all satellites are plotted. (default: None)
    y_log : bool, optional
        Use logarithmic scale for y-axis (Delta V). (default: True)
    x_axis_days : bool, optional
        If True, show x-axis in days. If False, show in hours. (default: False)
    """
    # Find all Octahedron CSV files
    csv_files = list(summaries_dir.glob("Octahedron_*.csv"))
    
    if not csv_files:
        print(f"No Octahedron CSV files found in {summaries_dir}")
        return
    
    # Initialize exclude list
    if exclude_names is None:
        exclude_names = []
    
    # Convert exclude list to uppercase for consistency
    exclude_names = [name.upper() for name in exclude_names]
    
    # Collect data organized by satellite
    satellite_data = defaultdict(lambda: {
        'sk_hours': [],
        'sk_display': [],
        'delta_v': [],
        'filenames': []  # Store filenames for filtering
    })
    
    # Process each file
    for csv_file in csv_files:
        filename = csv_file.stem
        sk_string, axis, sign = extract_satellite_info(filename)
        
        if sk_string is None:
            continue
        
        # Parse SK interval
        try:
            sk_hours, sk_display = parse_sk_interval(sk_string)
        except ValueError:
            continue
        
        # Extract Delta V
        delta_v = extract_delta_v_from_header(csv_file)
        if delta_v is None:
            continue
        
        # Create satellite key
        sat_key = f"{axis}{sign[0].upper()}"  # e.g., "XM", "YP", etc.
        
        satellite_data[sat_key]['sk_hours'].append(sk_hours)
        satellite_data[sat_key]['sk_display'].append(sk_display)
        satellite_data[sat_key]['delta_v'].append(delta_v)
        satellite_data[sat_key]['filenames'].append(filename)
    
    # Sort each satellite's data by SK hours
    for sat_key in satellite_data:
        indices = np.argsort(satellite_data[sat_key]['sk_hours'])
        satellite_data[sat_key]['sk_hours'] = np.array(satellite_data[sat_key]['sk_hours'])[indices]
        satellite_data[sat_key]['sk_display'] = np.array(satellite_data[sat_key]['sk_display'])[indices]
        satellite_data[sat_key]['delta_v'] = np.array(satellite_data[sat_key]['delta_v'])[indices]
        satellite_data[sat_key]['filenames'] = np.array(satellite_data[sat_key]['filenames'])[indices]
    
    # Filter satellite data by name patterns
    filtered_satellites = {}
    excluded_names_list = []
    
    for sat_key, data in satellite_data.items():
        filenames = data.get('filenames', [])
        
        # Keep only filenames that don't match any exclusion pattern
        filtered_indices = []
        
        for i, filename in enumerate(filenames):
            should_exclude = False
            
            for pattern in exclude_names:
                if pattern in filename.upper():
                    should_exclude = True
                    excluded_names_list.append(filename)
                    break
            
            if not should_exclude:
                filtered_indices.append(i)
        
        # Only include satellite type if it still has data points after filtering
        if filtered_indices:
            filtered_data = {}
            for key in data:
                if key == 'filenames':
                    filtered_data[key] = np.array(data[key])[filtered_indices]
                else:
                    filtered_data[key] = np.array(data[key])[filtered_indices]
            
            filtered_satellites[sat_key] = filtered_data
    
    if not filtered_satellites:
        print("No satellites to plot after applying exclusions")
        return
    
    # Print excluded satellites info
    if excluded_names_list:
        print(f"Excluding satellite patterns: {', '.join(exclude_names)}")
        print(f"Excluded satellites:")
        for name in sorted(set(excluded_names_list)):
            print(f"  - {name}")
        print()
    
    # Define colors: X=blue, Y=green, Z=red
    # Light for plus, dark for minus
    colors = {
        'XP': '#4DA6FF',  # Light blue
        'XM': '#0033CC',  # Dark blue
        'YP': '#66FF66',  # Light green
        'YM': '#006600',  # Dark green
        'ZP': '#FF6666',  # Light red
        'ZM': '#CC0000',  # Dark red
    }
    
    # Create figure
    fig, ax = plt.subplots(figsize=PLOT_DIM_SINGLE)
    
    # Collect all unique SK intervals for x-axis labels
    all_sk_displays = []
    all_sk_hours = []
    for sat_key in filtered_satellites.keys():
        all_sk_displays.extend(filtered_satellites[sat_key]['sk_display'].tolist())
        all_sk_hours.extend(filtered_satellites[sat_key]['sk_hours'].tolist())
    
    # Get unique SK intervals (preserving order)
    unique_sk_pairs = list(dict.fromkeys(zip(all_sk_hours, all_sk_displays)))
    unique_sk_hours_sorted = sorted(unique_sk_pairs, key=lambda x: x[0])
    
    # Prepare x-axis data and labels
    if x_axis_days:
        x_plot_data = np.array([sk_hours / 24.0 for sk_hours, _ in unique_sk_hours_sorted])
        x_label = 'Station Keeping Interval'
    else:
        x_plot_data = np.array([sk_hours for sk_hours, _ in unique_sk_hours_sorted])
        x_label = 'Station Keeping Interval'
    
    # Remove "SK" prefix from labels, keep only the time value
    x_tick_labels = [sk_display.replace('SK', '') for _, sk_display in unique_sk_hours_sorted]
    
    # Plot each satellite (only non-excluded ones)
    for sat_key in sorted(filtered_satellites.keys()):
        data = filtered_satellites[sat_key]
        color = colors.get(sat_key, '#000000')
        
        # Convert x-axis data if needed
        if x_axis_days:
            x_data = np.array(data['sk_hours']) / 24.0
        else:
            x_data = np.array(data['sk_hours'])
        
        ax.plot(x_data, data['delta_v'] * 1000,  # Convert to mm/s for better scale
                marker='o', label=sat_key, color=color, linewidth=2, markersize=8)
    
    # Formatting
    ax.set_xlabel(x_label, fontsize=12, fontweight='bold')
    ax.set_ylabel('Total Delta V (mm/s/orbit)', fontsize=12, fontweight='bold')
    ax.set_title('Delta V Required vs Station Keeping Duration', 
                 fontsize=14, fontweight='bold')
    
    ax.grid(True, alpha=0.3, linestyle='--')
    ax.legend(ncol=2, loc='best', fontsize=10, framealpha=0.9)
    
    # Set x-axis to log scale and use custom tick labels
    ax.set_xscale('log')
    ax.set_xticks(x_plot_data)
    ax.set_xticklabels(x_tick_labels, ha='center')
    
    # Set y-axis scale based on parameter
    if y_log:
        ax.set_yscale('log')
    
    plt.tight_layout()
    
    # Save figure
    plots_dir = current_dir / "results"
    plots_dir.mkdir(exist_ok=True)
    
    # Add scale and exclusion info to filename if applicable
    filename = "delta_v_vs_sk_duration"
    if x_axis_days:
        filename += "_xdays"
    if y_log:
        filename += "_ylog"
    if excluded_names_list:
        exclude_suffix = "_exclude_" + "_".join([name.split('_')[1] for name in excluded_names_list]).lower()
        filename += exclude_suffix
    filename += ".png"
    
    fig.savefig(plots_dir / filename, dpi=600, bbox_inches='tight')
    print(f"Plot saved to: {plots_dir / filename}")
    
    plt.show()


## 11. Define the intended-offset decoder

`extract_offset_from_filename` turns a direction/size token into a Cartesian offset in kilometres. For example, `Xp1000m` gives `(1, 0, 0)` km, and `Ym1000m` gives `(0, -1, 0)` km.

The preserved pattern expects uppercase axis letters, lowercase `p`/`m`, and an integer size in metres. These offsets come from names rather than report metadata. They must describe the same axes and vector direction as the input position columns.

In [12]:
def extract_offset_from_filename(filename):
    """
    Extract the intended offset from the filename.
    
    Example: "Octahedron_SK7d_Xp1000m" -> {'x': 1.0, 'y': 0.0, 'z': 0.0}
    """
    parts = filename.split('_')
    if len(parts) < 3:
        return None
    
    direction_part = parts[2]  # e.g., "Xp1000m"
    
    # Parse direction: Xp1000m, Xm1000m, etc.
    match = re.match(r'([XYZ])([pm])(\d+)m', direction_part)
    if not match:
        return None
    
    axis = match.group(1)
    sign = 1.0 if match.group(2) == 'p' else -1.0
    magnitude_m = float(match.group(3))
    
    # Convert meters to km
    magnitude_km = magnitude_m / 1000.0
    
    offset = {'x': 0.0, 'y': 0.0, 'z': 0.0}
    offset[axis.lower()] = sign * magnitude_km
    
    return offset


## 12. Define deviations from the intended offset

`shift_position_data_to_origin` reads the position table after `--- BEGIN DATA ---`, copies it, and subtracts the intended offset from x, y and z. It adds four columns, all in kilometres:

$$\Delta x=x-x_0,\quad \Delta y=y-y_0,\quad \Delta z=z-z_0,$$
$$\Delta r=\sqrt{(\Delta x)^2+(\Delta y)^2+(\Delta z)^2}.$$

The magnitude is the distance from the intended position, not the difference between two scalar distances. Original columns remain in the copied table.

STK's [Vector Choose Axes reference](https://help.agi.com/stkdevkit/12.6.0/Subsystems/dataProviders/Content/html/dataProviders/Vector_Choose_Axes.htm) defines components in the axes selected for the report. This function does not rotate coordinates or reverse vector signs; input positions must already use the desired frame and satellite-relative-to-Base convention. The exact custom frame cannot be inferred from this file alone.

In [13]:
def shift_position_data_to_origin(csv_file_path, offset):
    """
    Shift position data to the origin of the satellite by subtracting the offset.
    Returns a dataframe with the shifted positions.
    
    Parameters:
    -----------
    csv_file_path : Path
        Path to the CSV position file
    offset : dict
        Dictionary with 'x', 'y', 'z' keys (in km)
    
    Returns:
    --------
    df_deviations : pd.DataFrame
        DataFrame with deviation columns
    """
    # Find the data start marker
    with open(csv_file_path, 'r') as f:
        lines = f.readlines()
    
    data_start_idx = None
    for i, line in enumerate(lines):
        if '--- BEGIN DATA ---' in line:
            data_start_idx = i + 1
            break
    
    if data_start_idx is None:
        raise ValueError("Could not find '--- BEGIN DATA ---' marker in file")
    
    # Read the CSV starting from the data section
    df = pd.read_csv(csv_file_path, skiprows=data_start_idx)
    
    # Create deviation columns by subtracting offset
    df_deviations = df.copy()
    df_deviations['x_deviation (km)'] = df['x (km)'] - offset['x']
    df_deviations['y_deviation (km)'] = df['y (km)'] - offset['y']
    df_deviations['z_deviation (km)'] = df['z (km)'] - offset['z']
    
    # Calculate magnitude of deviation
    df_deviations['magnitude_deviation (km)'] = np.sqrt(
        df_deviations['x_deviation (km)']**2 + 
        df_deviations['y_deviation (km)']**2 + 
        df_deviations['z_deviation (km)']**2
    )
    
    return df_deviations


## 13. Define the deviation CSV writer

`save_deviation_data` creates a sibling `deviations` directory beside the original `positions` directory and writes `<original_stem>_deviations.csv` there.

The saved table includes the copied original columns and the four new deviation columns, with a standard CSV column-header row and no metadata preamble or Pandas index. The source comment “without header rows” refers to removing the metadata preamble; column names are still written. The function returns the new path.

In [14]:
def save_deviation_data(csv_file_path, df_deviations):
    """
    Save deviation data to a new CSV file with _deviations suffix.
    
    Parameters:
    -----------
    csv_file_path : Path
        Original CSV path
    df_deviations : pd.DataFrame
        DataFrame with deviation columns
    
    Returns:
    --------
    output_path : Path
        Path to the saved deviation file
    """
    csv_path = Path(csv_file_path)
    
    # Create deviations folder
    deviations_dir = csv_path.parent.parent / "deviations"
    deviations_dir.mkdir(parents=True, exist_ok=True)
    
    # Create new filename with _deviations suffix
    stem = csv_path.stem
    output_filename = f"{stem}_deviations.csv"
    output_path = deviations_dir / output_filename
    
    # Save CSV (without header rows, just data)
    df_deviations.to_csv(output_path, index=False)
    
    return output_path


## 14. Define the maximum-deviation analysis

`calculate_max_deviations` scans `data-export/positions/Octahedron_*.csv`, decodes each intended offset, computes deviations, and saves each deviation table. It then finds the largest absolute x, y and z deviations, and the largest deviation-vector magnitude, over the available rows.

Its return value is `satellite_data`: arrays grouped by axis/sign and sorted by SK interval. It stores full filenames for later exclusions. All stored distances remain in kilometres; plotting converts them to metres.

These are maxima over sampled report rows, so peaks between samples may be missed. Component maxima can occur at different times; the magnitude maximum is computed separately from the instantaneous vector norm. Processing errors are printed and skipped. This calculation itself applies no exclusion list.

In [15]:
def calculate_max_deviations():
    """
    Calculate maximum deviations for all position CSV files.
    
    Returns:
    --------
    dict : Dictionary organized by satellite with SK intervals and max deviations
    """
    current_dir = Path.cwd()
    positions_dir = current_dir / "data-export" / "positions"
    
    csv_files = list(positions_dir.glob("Octahedron_*.csv"))
    
    if not csv_files:
        print(f"No position CSV files found in {positions_dir}")
        return {}
    
    # Organize data by satellite
    satellite_data = defaultdict(lambda: {
        'full_names': [],  # Store full satellite names
        'sk_hours': [],
        'sk_display': [],
        'max_x_dev': [],
        'max_y_dev': [],
        'max_z_dev': [],
        'max_mag_dev': []
    })
    
    print(f'\n{"="*100}')
    print(f'CALCULATING POSITION DEVIATIONS')
    print(f'{"="*100}\n')
    
    for csv_file in sorted(csv_files):
        filename = csv_file.stem
        
        # Extract SK interval and satellite direction
        parts = filename.split('_')
        if len(parts) < 3:
            continue
        
        sk_string = parts[1]  # e.g., "SK7d"
        direction_part = parts[2]  # e.g., "Xp1000m"
        
        # Extract offset
        offset = extract_offset_from_filename(filename)
        if offset is None:
            print(f"  ✗ Could not parse offset from {filename}")
            continue
        
        # Parse SK interval
        try:
            sk_hours, sk_display = parse_sk_interval(sk_string)
        except ValueError as e:
            print(f"  ✗ {filename}: {e}")
            continue
        
        # Calculate deviations
        try:
            df_deviations = shift_position_data_to_origin(csv_file, offset)
            
            # Save deviation data
            output_path = save_deviation_data(csv_file, df_deviations)
            
            # Calculate maximum deviations
            max_x_dev = df_deviations['x_deviation (km)'].abs().max()
            max_y_dev = df_deviations['y_deviation (km)'].abs().max()
            max_z_dev = df_deviations['z_deviation (km)'].abs().max()
            max_mag_dev = df_deviations['magnitude_deviation (km)'].max()
            
            # Create satellite key (axis and direction)
            match = re.match(r'([XYZ])([pm])', direction_part)
            if match:
                axis = match.group(1)
                sign = 'P' if match.group(2) == 'p' else 'M'
                sat_key = f"{axis}{sign}"
                
                satellite_data[sat_key]['full_names'].append(filename)  # Store full name
                satellite_data[sat_key]['sk_hours'].append(sk_hours)
                satellite_data[sat_key]['sk_display'].append(sk_string)
                satellite_data[sat_key]['max_x_dev'].append(max_x_dev)
                satellite_data[sat_key]['max_y_dev'].append(max_y_dev)
                satellite_data[sat_key]['max_z_dev'].append(max_z_dev)
                satellite_data[sat_key]['max_mag_dev'].append(max_mag_dev)
                
                print(f"  ✓ {filename}")
                print(f"      Max deviations - X: {max_x_dev*1000:.3f}m, Y: {max_y_dev*1000:.3f}m, Z: {max_z_dev*1000:.3f}m, Mag: {max_mag_dev*1000:.3f}m")
                
        except Exception as e:
            print(f"  ✗ {filename}: Error processing - {str(e)}")
            continue
    
    # Sort each satellite's data by SK hours
    for sat_key in satellite_data:
        indices = np.argsort(satellite_data[sat_key]['sk_hours'])
        for key in ['full_names', 'sk_hours', 'sk_display', 'max_x_dev', 'max_y_dev', 'max_z_dev', 'max_mag_dev']:
            satellite_data[sat_key][key] = np.array(satellite_data[sat_key][key])[indices]
    
    return satellite_data


## 15. Define the four-panel maximum-deviation plot

`plot_max_deviations` plots maximum absolute x, y and z deviations and maximum vector-magnitude deviation in a 2 × 2 figure. It uses `satellite_data` from the previous function, applies substring exclusions, and converts kilometres to metres.

SK coordinates are in days. `x_log` and `y_log` select logarithmic scales; custom hour/day tick labels are set when `x_log=True`. Each curve represents an axis/sign group.

The figure is saved to `results/position_deviations_vs_sk...png` before display. Logarithmic axes require positive plotted values; zero deviations will not be represented normally on a log y-axis.

In [16]:
def plot_max_deviations(satellite_data, x_log=False, y_log=False, exclude_names=None):
    """
    Plot maximum deviations vs SK interval in 4 separate plots (X, Y, Z, Magnitude).
    
    Parameters:
    -----------
    satellite_data : dict
        Dictionary with deviation data organized by satellite
    x_log : bool, optional
        Use logarithmic scale for x-axis (SK interval) (default: False)
    y_log : bool, optional
        Use logarithmic scale for y-axis (deviation) (default: False)
    exclude_names : list, optional
        List of satellite name patterns to exclude from the plot.
        Matches against full satellite names using substring matching (case-insensitive).
        Examples:
            ['SK3d']              - Exclude all satellites with 'SK3d' in name
            ['Octahedron_SK1h_Ym1000m']  - Exclude specific satellite by full name
            ['SK3d', 'Ym']        - Exclude SK3d satellites OR Ym satellites
        If None, all satellites are plotted. (default: None)
    """
    if not satellite_data:
        print("No satellite data to plot")
        return
    
    # Initialize exclude list
    if exclude_names is None:
        exclude_names = []
    
    # Convert exclude list to uppercase for consistency
    exclude_names = [name.upper() for name in exclude_names]
    
    # Filter satellite data by name patterns
    filtered_satellites = {}
    excluded_names_list = []
    
    for sat_key, data in satellite_data.items():
        # Get full names for this satellite type
        full_names = data.get('full_names', [])
        
        # Keep only full names that don't match any exclusion pattern
        filtered_full_names = []
        
        for full_name in full_names:
            should_exclude_this = False
            
            for pattern in exclude_names:
                if pattern in full_name.upper():
                    should_exclude_this = True
                    excluded_names_list.append(full_name)
                    break
            
            if not should_exclude_this:
                filtered_full_names.append(full_name)
        
        # Only include satellite type if it still has data points after filtering
        if filtered_full_names:
            # Create a copy of data with filtered entries
            filtered_data = {}
            for key in data:
                if key == 'full_names':
                    filtered_data[key] = np.array(filtered_full_names)
                else:
                    # Get indices of kept satellites
                    indices = [i for i, name in enumerate(full_names) if name in filtered_full_names]
                    filtered_data[key] = np.array(data[key])[indices]
            
            filtered_satellites[sat_key] = filtered_data
    
    if not filtered_satellites:
        print("No satellites to plot after applying exclusions")
        return
    
    # Print excluded satellites info
    if excluded_names_list:
        print(f"Excluding satellite patterns: {', '.join(exclude_names)}")
        print(f"Excluded satellites:")
        for name in sorted(set(excluded_names_list)):
            print(f"  - {name}")
        print(f"Plotting remaining data points")
    
    # Define colors: X=blue, Y=green, Z=red
    colors = {
        'XP': '#4DA6FF',  # Light blue
        'XM': '#0033CC',  # Dark blue
        'YP': '#66FF66',  # Light green
        'YM': '#006600',  # Dark green
        'ZP': '#FF6666',  # Light red
        'ZM': '#CC0000',  # Dark red
    }
    
    # Create 4 subplots
    fig, axes = plt.subplots(2, 2, figsize=PLOT_DIM_4SUBPLOT)
    axes = axes.flatten()
    
    # Collect all unique SK intervals for custom x-axis labels
    all_sk_displays = []
    all_sk_hours = []
    for sat_key in filtered_satellites.keys():
        all_sk_displays.extend(filtered_satellites[sat_key]['sk_display'].tolist())
        all_sk_hours.extend(filtered_satellites[sat_key]['sk_hours'].tolist())
    
    # Get unique SK intervals (preserving order)
    unique_sk_pairs = list(dict.fromkeys(zip(all_sk_hours, all_sk_displays)))
    unique_sk_hours_sorted = sorted(unique_sk_pairs, key=lambda x: x[0])
    
    # Prepare x-axis data and labels in days
    x_plot_data = np.array([sk_hours / 24.0 for sk_hours, _ in unique_sk_hours_sorted])
    x_tick_labels = [sk_display.replace('SK', '') for _, sk_display in unique_sk_hours_sorted]
    
    plot_configs = [
        ('max_x_dev', 'X Deviation (m)', axes[0]),
        ('max_y_dev', 'Y Deviation (m)', axes[1]),
        ('max_z_dev', 'Z Deviation (m)', axes[2]),
        ('max_mag_dev', 'Magnitude Deviation (m)', axes[3])
    ]
    
    for data_key, ylabel, ax in plot_configs:
        # Plot each satellite (only non-excluded ones)
        for sat_key in sorted(filtered_satellites.keys()):
            data = filtered_satellites[sat_key]
            color = colors.get(sat_key, '#000000')
            
            # Convert SK hours to days
            sk_days = np.array(data['sk_hours']) / 24.0
            
            # Convert deviations to meters
            y_values = np.array(data[data_key]) * 1000
            
            ax.plot(sk_days, y_values,
                    marker='o', label=sat_key, color=color, linewidth=2, markersize=8)
        
        # Formatting
        ax.set_xlabel('Station Keeping Interval', fontsize=11, fontweight='bold')
        ax.set_ylabel(ylabel, fontsize=11, fontweight='bold')
        ax.grid(True, alpha=0.3, linestyle='--')
        ax.legend(ncol=2, loc='best', fontsize=9, framealpha=0.9)
        
        # Set x-axis to log scale and use custom tick labels
        if x_log:
            ax.set_xscale('log')
            ax.set_xticks(x_plot_data)
            ax.set_xticklabels(x_tick_labels, ha='center')
        if y_log:
            ax.set_yscale('log')
    
    fig.suptitle('Maximum Position Deviations from Intended Offset\n(Octahedron Constellation)', 
                 fontsize=14, fontweight='bold', y=0.995)
    plt.tight_layout()
    
    # Save figure
    current_dir = Path.cwd()
    plots_dir = current_dir / "results"
    plots_dir.mkdir(exist_ok=True)
    
    # Add scale info to filename
    scale_suffix = ""
    if x_log:
        scale_suffix += "_xlog"
    if y_log:
        scale_suffix += "_ylog"
    if excluded_names_list:
        exclude_suffix = "_exclude_" + "_".join([name.split('_')[1] for name in excluded_names_list]).lower()
        scale_suffix += exclude_suffix
    
    filename = f"position_deviations_vs_sk{scale_suffix}.png"
    fig.savefig(plots_dir / filename, dpi=600, bbox_inches='tight')
    print(f"Plot saved to: {plots_dir / filename}")
    
    plt.show()


## 16. Define the deviation time-series plot

`plot_deviations_over_time` reads one generated deviation CSV and plots signed x, y and z deviations plus the nonnegative vector-magnitude deviation in four stacked panels. It converts values to metres, adds summary statistics, prints a summary, saves `results/deviations_vs_days_<case>.png` or its `_vs_hours_` counterpart, and displays it. Standard deviation uses Pandas' sample-standard-deviation default.

With `x_axis_days=True`, time is constructed from row number assuming a constant 10-minute report step. It does not use the timestamps. Gaps, duplicate times or different sampling would invalidate that constructed time axis.

**Preserved issue:** with `x_axis_days=False`, the function uses row numbers directly and labels them hours. This is inaccurate for 10-minute samples. Every active time-series call later in this notebook uses `x_axis_days=True`. Missing files are listed and skipped; this function does not apply the comparison plots' exclusion lists.

In [17]:
def plot_deviations_over_time(satellite_name, sk_interval=None, x_axis_days=False):
    """
    Plot position deviations (X, Y, Z, and Magnitude) over time for a specified satellite.
    
    Parameters:
    -----------
    satellite_name : str
        Satellite name (e.g., "Octahedron_SK7d_Xp1000m")
    sk_interval : str, optional
        SK interval to filter (e.g., "SK7d"). If None, uses the satellite_name as-is.
        (default: None)
    x_axis_days : bool, optional
        If True, show x-axis in days.
        If False, show x-axis in hours from start. (default: False)
    
    Example:
    --------
    # Plot specific satellite with time in hours
    plot_deviations_over_time("Octahedron_SK7d_Xp1000m")
    
    # Plot with x-axis in days
    plot_deviations_over_time("Octahedron_SK7d_Xp1000m", x_axis_days=True)
    
    # Or search by SK interval and satellite type
    plot_deviations_over_time("Xp1000m", sk_interval="SK7d", x_axis_days=True)
    """
    # Enable interactive mode for zooming/panning
    plt.ion()
    
    current_dir = Path.cwd()
    deviations_dir = current_dir / "data-export" / "deviations"
    
    # Build the filename
    if sk_interval:
        # Search mode: construct filename from components
        deviation_filename = f"Octahedron_{sk_interval}_{satellite_name}_deviations.csv"
    else:
        # Direct mode: use satellite_name as full filename
        deviation_filename = f"{satellite_name}_deviations.csv"
    
    deviation_path = deviations_dir / deviation_filename
    
    if not deviation_path.exists():
        print(f"✗ File not found: {deviation_path}")
        print(f"\nAvailable files in {deviations_dir}:")
        available = list(deviations_dir.glob("*.csv"))
        for f in sorted(available)[:10]:  # Show first 10
            print(f"  - {f.name}")
        if len(available) > 10:
            print(f"  ... and {len(available) - 10} more")
        return
    
    # Read deviation data
    try:
        df = pd.read_csv(deviation_path)
    except Exception as e:
        print(f"✗ Error reading file: {str(e)}")
        return
    
    # Extract the satellite info from filename for title
    title_name = deviation_path.stem.replace('_deviations', '')
    
    # Create figure with 4 subplots
    fig, axes = plt.subplots(4, 1, figsize=PLOT_DIM_TIMESERIES)
    
    # Prepare x-axis data
    if x_axis_days:
        # Convert hours to days (using 10-minute time steps from export)
        time_step_hours = 10.0 / 60.0  # 10 minutes = 0.1667 hours
        x_axis_data = np.arange(len(df)) * time_step_hours / 24.0
        x_label = 'Time (days from start)'
    else:
        # Time in hours from start
        x_axis_data = np.arange(len(df))
        x_label = 'Time (hours from start)'
    
    # Plot configurations
    plot_configs = [
        ('x_deviation (km)', r'$\Delta x$ (m)', axes[0], '#4DA6FF'),
        ('y_deviation (km)', r'$\Delta y$ (m)', axes[1], '#66FF66'),
        ('z_deviation (km)', r'$\Delta z$ (m)', axes[2], '#FF6666'),
        ('magnitude_deviation (km)', r'Magnitude (m)', axes[3], '#FF9900')
    ]
    
    for col_name, ylabel, ax, color in plot_configs:
        if col_name not in df.columns:
            print(f"✗ Column '{col_name}' not found in CSV")
            continue
        
        # Convert to meters
        y_values = df[col_name] * 1000
        
        ax.plot(x_axis_data, y_values, color=color, linewidth=1.5, alpha=0.8)
        ax.fill_between(x_axis_data, y_values, alpha=0.2, color=color)
        
        # Add statistics
        max_dev = y_values.max()
        min_dev = y_values.min()
        mean_dev = y_values.mean()
        std_dev = y_values.std()
        
        ax.axhline(mean_dev, color='red', linestyle='--', linewidth=1, alpha=0.5, label=f'Mean: {mean_dev:.3f}m')
        ax.axhline(max_dev, color='darkgreen', linestyle=':', linewidth=1, alpha=0.5, label=f'Max: {max_dev:.3f}m')
        
        # Formatting
        ax.set_ylabel(ylabel, fontsize=11, fontweight='bold')
        ax.grid(True, alpha=0.3, linestyle='--')
        ax.legend(loc='upper right', fontsize=9)
        
        # Add statistics text box
        stats_text = f'Max: {max_dev:.3f}m\nMin: {min_dev:.3f}m\nMean: {mean_dev:.3f}m\nStd: {std_dev:.3f}m'
        ax.text(0.02, 0.98, stats_text, transform=ax.transAxes, 
                fontsize=9, verticalalignment='top',
                bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
    
    axes[-1].set_xlabel(x_label, fontsize=11, fontweight='bold')
    
    axis_type = 'Days' if x_axis_days else 'Hours'
    fig.suptitle(f'Position Deviations Over {axis_type}\n{title_name}', 
                 fontsize=14, fontweight='bold', y=0.995)
    plt.tight_layout()
    
    # Save figure
    plots_dir = current_dir / "results"
    plots_dir.mkdir(exist_ok=True)
    
    axis_suffix = '_vs_days' if x_axis_days else '_vs_hours'
    filename = f"deviations{axis_suffix}_{title_name}.png"
    fig.savefig(plots_dir / filename, dpi=600, bbox_inches='tight')
    print(f"\n✓ Plot saved to: {plots_dir / filename}")
    
    # Print summary
    print(f"\nDeviation Summary for {title_name}:")
    for col_name, ylabel, ax, color in plot_configs:
        if col_name in df.columns:
            y_values = df[col_name] * 1000
            print(f"  {ylabel}:")
            print(f"    Max: {y_values.max():.3f}m, Min: {y_values.min():.3f}m, Mean: {y_values.mean():.3f}m")
    
    plt.show()


## 17. Define the magnitude-only comparison

`plot_magnitude_deviations` extracts `max_mag_dev` from `satellite_data` and plots it against the SK interval for each axis/sign group. It converts kilometre deviations to metres and hour intervals to days.

The optional exclusions and logarithmic scales work like those in the four-panel plot. This view highlights the largest distance from each satellite's intended offset, rather than individual Cartesian components. It saves `results/magnitude_deviation_vs_sk...png` and displays the figure.

In [18]:
def plot_magnitude_deviations(satellite_data, x_log=False, y_log=False, exclude_names=None):
    """
    Plot maximum magnitude deviations vs SK interval.
    
    Parameters:
    -----------
    satellite_data : dict
        Dictionary with deviation data organized by satellite
    x_log : bool, optional
        Use logarithmic scale for x-axis (SK interval) (default: False)
    y_log : bool, optional
        Use logarithmic scale for y-axis (deviation) (default: False)
    exclude_names : list, optional
        List of satellite name patterns to exclude from the plot.
        Matches against full satellite names using substring matching (case-insensitive).
        Examples:
            ['SK3d']              - Exclude all satellites with 'SK3d' in name
            ['Octahedron_SK1h_Ym1000m']  - Exclude specific satellite by full name
            ['SK3d', 'Ym']        - Exclude SK3d satellites OR Ym satellites
        If None, all satellites are plotted. (default: None)
    """
    if not satellite_data:
        print("No satellite data to plot")
        return
    
    # Initialize exclude list
    if exclude_names is None:
        exclude_names = []
    
    # Convert exclude list to uppercase for consistency
    exclude_names = [name.upper() for name in exclude_names]
    
    # Filter satellite data by name patterns
    filtered_satellites = {}
    excluded_names_list = []
    
    for sat_key, data in satellite_data.items():
        # Get full names for this satellite type
        full_names = data.get('full_names', [])
        
        # Keep only full names that don't match any exclusion pattern
        filtered_full_names = []
        
        for full_name in full_names:
            should_exclude_this = False
            
            for pattern in exclude_names:
                if pattern in full_name.upper():
                    should_exclude_this = True
                    excluded_names_list.append(full_name)
                    break
            
            if not should_exclude_this:
                filtered_full_names.append(full_name)
        
        # Only include satellite type if it still has data points after filtering
        if filtered_full_names:
            # Create a copy of data with filtered entries
            filtered_data = {}
            for key in data:
                if key == 'full_names':
                    filtered_data[key] = np.array(filtered_full_names)
                else:
                    # Get indices of kept satellites
                    indices = [i for i, name in enumerate(full_names) if name in filtered_full_names]
                    filtered_data[key] = np.array(data[key])[indices]
            
            filtered_satellites[sat_key] = filtered_data
    
    if not filtered_satellites:
        print("No satellites to plot after applying exclusions")
        return
    
    # Print excluded satellites info
    if excluded_names_list:
        print(f"Excluding satellite patterns: {', '.join(exclude_names)}")
        print(f"Excluded satellites:")
        for name in sorted(set(excluded_names_list)):
            print(f"  - {name}")
        print(f"Plotting remaining data points")
    
    # Define colors: X=blue, Y=green, Z=red
    colors = {
        'XP': '#4DA6FF',  # Light blue
        'XM': '#0033CC',  # Dark blue
        'YP': '#66FF66',  # Light green
        'YM': '#006600',  # Dark green
        'ZP': '#FF6666',  # Light red
        'ZM': '#CC0000',  # Dark red
    }
    
    # Create single subplot for magnitude only
    fig, ax = plt.subplots(figsize=PLOT_DIM_SINGLE)
    
    # Collect all unique SK intervals for custom x-axis labels
    all_sk_displays = []
    all_sk_hours = []
    for sat_key in filtered_satellites.keys():
        all_sk_displays.extend(filtered_satellites[sat_key]['sk_display'].tolist())
        all_sk_hours.extend(filtered_satellites[sat_key]['sk_hours'].tolist())
    
    # Get unique SK intervals (preserving order)
    unique_sk_pairs = list(dict.fromkeys(zip(all_sk_hours, all_sk_displays)))
    unique_sk_hours_sorted = sorted(unique_sk_pairs, key=lambda x: x[0])
    
    # Prepare x-axis data and labels in days
    x_plot_data = np.array([sk_hours / 24.0 for sk_hours, _ in unique_sk_hours_sorted])
    x_tick_labels = [sk_display.replace('SK', '') for _, sk_display in unique_sk_hours_sorted]
    
    # Plot each satellite (only non-excluded ones)
    for sat_key in sorted(filtered_satellites.keys()):
        data = filtered_satellites[sat_key]
        color = colors.get(sat_key, '#000000')
        
        # Convert SK hours to days
        sk_days = np.array(data['sk_hours']) / 24.0
        
        # Convert deviations to meters
        y_values = np.array(data['max_mag_dev']) * 1000
        
        ax.plot(sk_days, y_values,
                marker='o', label=sat_key, color=color, linewidth=2, markersize=8)
    
    # Formatting
    ax.set_xlabel('Station Keeping Interval', fontsize=12, fontweight='bold')
    ax.set_ylabel(r'$\Delta r$ (m)', fontsize=12, fontweight='bold')
    ax.set_title('Maximum Magnitude Deviation from Intended Offset\n(Octahedron Constellation)', 
                 fontsize=14, fontweight='bold')
    ax.grid(True, alpha=0.3, linestyle='--')
    ax.legend(ncol=2, loc='best', fontsize=10, framealpha=0.9)
    
    # Set x-axis to log scale and use custom tick labels
    if x_log:
        ax.set_xscale('log')
        ax.set_xticks(x_plot_data)
        ax.set_xticklabels(x_tick_labels, ha='center')
    if y_log:
        ax.set_yscale('log')
    
    plt.tight_layout()
    
    # Save figure
    current_dir = Path.cwd()
    plots_dir = current_dir / "results"
    plots_dir.mkdir(exist_ok=True)
    
    # Add scale info to filename
    scale_suffix = ""
    if x_log:
        scale_suffix += "_xlog"
    if y_log:
        scale_suffix += "_ylog"
    if excluded_names_list:
        exclude_suffix = "_exclude_" + "_".join([name.split('_')[1] for name in excluded_names_list]).lower()
        scale_suffix += exclude_suffix
    
    filename = f"magnitude_deviation_vs_sk{scale_suffix}.png"
    fig.savefig(plots_dir / filename, dpi=600, bbox_inches='tight')
    print(f"Plot saved to: {plots_dir / filename}")
    
    plt.show()


## 18. Define the Δv–deviation trade-off plot

`plot_delta_v_vs_max_deviation` joins deviation results and maneuver-header Δv totals by exact case filename. It excludes requested substrings, groups the matched cases by SK interval, and plots one point per interval.

The x-coordinate is the arithmetic mean of the matched satellites' individual maximum magnitude deviations, converted to metres. The y-coordinate is their mean Δv, converted to mm/s. This is a mean of maxima, not the worst satellite's maximum, and mean Δv is not constellation-total Δv. Missing matching records reduce the averaging sample; the function does not require all six satellites.

The `mm/s/orbit` label again assumes the inputs already cover one orbit; no orbit normalisation is performed here. Comparisons need consistent report durations and satellite sets. The function saves `results/delta_v_vs_max_deviation...png`, with scale and exclusion suffixes, then displays it.

In [19]:
def plot_delta_v_vs_max_deviation(satellite_data, delta_v_data=None, x_log=False, y_log=False, exclude_names=None):
    """
    Plot Delta V vs Maximum Magnitude Deviation, with different colors for each SK interval.
    Data points are averaged across satellites for each SK interval.
    
    Parameters:
    -----------
    satellite_data : dict
        Dictionary with deviation data organized by satellite (from calculate_max_deviations)
    delta_v_data : dict, optional
        Pre-calculated Delta V data. If None, will be calculated from CSV files. (default: None)
    x_log : bool, optional
        Use logarithmic scale for x-axis (deviation). (default: False)
    y_log : bool, optional
        Use logarithmic scale for y-axis (Delta V). (default: False)
    exclude_names : list, optional
        List of satellite name patterns to exclude from the plot.
        Matches against full satellite names using substring matching (case-insensitive).
        Examples:
            ['SK3d']              - Exclude all satellites with 'SK3d' in name
            ['SK3d', 'Ym']        - Exclude SK3d satellites OR Ym satellites
        If None, all satellites are plotted. (default: None)
    """
    if not satellite_data:
        print("No satellite data to plot")
        return
    
    # Initialize exclude list
    if exclude_names is None:
        exclude_names = []
    
    # Convert exclude list to uppercase for consistency
    exclude_names = [name.upper() for name in exclude_names]
    
    # If delta_v_data not provided, calculate it from CSV files
    if delta_v_data is None:
        delta_v_data = {}
        csv_files = list(summaries_dir.glob("Octahedron_*.csv"))
        
        for csv_file in csv_files:
            filename = csv_file.stem
            delta_v = extract_delta_v_from_header(csv_file)
            if delta_v is not None:
                delta_v_data[filename] = delta_v
    
    # Build a mapping of satellite to SK interval and Delta V
    sat_to_sk_and_dv = {}
    
    for sat_key, data in satellite_data.items():
        full_names = data.get('full_names', [])
        sk_displays = data.get('sk_display', [])
        max_mag_devs = data.get('max_mag_dev', [])
        
        for i, full_name in enumerate(full_names):
            # Check if should exclude
            should_exclude = False
            for pattern in exclude_names:
                if pattern in full_name.upper():
                    should_exclude = True
                    break
            
            if not should_exclude:
                sk_interval = sk_displays[i]
                max_dev = max_mag_devs[i]
                delta_v = delta_v_data.get(full_name, None)
                
                if delta_v is not None:
                    sat_to_sk_and_dv[full_name] = {
                        'sk_interval': sk_interval,
                        'max_dev': max_dev,
                        'delta_v': delta_v
                    }
    
    if not sat_to_sk_and_dv:
        print("No valid data after applying exclusions")
        return
    
    # Group by SK interval and calculate averages
    sk_groups = defaultdict(lambda: {'max_devs': [], 'delta_vs': []})
    
    for sat_name, info in sat_to_sk_and_dv.items():
        sk_interval = info['sk_interval']
        sk_groups[sk_interval]['max_devs'].append(info['max_dev'])
        sk_groups[sk_interval]['delta_vs'].append(info['delta_v'])
    
    # Calculate means for each SK interval
    sk_interval_data = {}
    for sk_interval, values in sk_groups.items():
        mean_dev = np.mean(values['max_devs'])
        mean_dv = np.mean(values['delta_vs'])
        sk_interval_data[sk_interval] = {
            'mean_dev': mean_dev,
            'mean_dv': mean_dv,
            'n_points': len(values['max_devs'])
        }
    
    # Sort SK intervals by hours for consistent coloring
    sk_hours_map = {}
    for sk_interval in sk_interval_data.keys():
        try:
            sk_hours, _ = parse_sk_interval(sk_interval)
            sk_hours_map[sk_interval] = sk_hours
        except ValueError:
            sk_hours_map[sk_interval] = float('inf')
    
    sorted_sk_intervals = sorted(sk_interval_data.keys(), key=lambda x: sk_hours_map[x])
    
    # Define color palette for SK intervals
    num_sk = len(sorted_sk_intervals)
    colors_sk = plt.cm.tab20(np.linspace(0, 1, num_sk))
    
    # Create figure
    fig, ax = plt.subplots(figsize=PLOT_DIM_SINGLE)
    
    # Plot each SK interval as a point
    for idx, sk_interval in enumerate(sorted_sk_intervals):
        data = sk_interval_data[sk_interval]
        color = colors_sk[idx]
        
        # Clean up SK interval label (remove 'SK' prefix)
        label = sk_interval.replace('SK', '')
        
        ax.scatter(data['mean_dev'] * 1000,  # Convert to meters
                   data['mean_dv'] * 1000,   # Convert to mm/s
                   s=200, color=color, label=label, alpha=0.7, edgecolors='black', linewidth=1.5)
    
    # Formatting
    ax.set_xlabel('Mean Maximum Deviation (m)', fontsize=12, fontweight='bold')
    ax.set_ylabel('Mean Delta V (mm/s/orbit)', fontsize=12, fontweight='bold')
    ax.set_title('Delta V vs Maximum Deviation by Station Keeping Interval\n(Octahedron Constellation)', 
                 fontsize=14, fontweight='bold')
    
    ax.grid(True, alpha=0.3, linestyle='--')
    ax.legend(title='SK Interval', ncol=1, loc='best', fontsize=10, framealpha=0.9, title_fontsize=11)
    
    # Set scales based on parameters
    if x_log:
        ax.set_xscale('log')
    if y_log:
        ax.set_yscale('log')
    
    plt.tight_layout()
    
    # Save figure
    current_dir = Path.cwd()
    plots_dir = current_dir / "results"
    plots_dir.mkdir(exist_ok=True)
    
    # Add scale info to filename
    scale_suffix = ""
    if x_log:
        scale_suffix += "_xlog"
    if y_log:
        scale_suffix += "_ylog"
    if exclude_names:
        exclude_suffix = "_exclude_" + "_".join(exclude_names).lower()
        scale_suffix += exclude_suffix
    
    filename = f"delta_v_vs_max_deviation{scale_suffix}.png"
    fig.savefig(plots_dir / filename, dpi=600, bbox_inches='tight')
    print(f"Plot saved to: {plots_dir / filename}")
    
    plt.show()


## 19. Generate the Δv versus SK interval figure

This cell creates `excluded_sats_for_delta_v_plot` and calls the Δv plotting function with logarithmic x and y scales and x-coordinates in days.

The original exclusions and comments are preserved. They describe the source author's noted simulation issues or missing cases; this notebook does not independently diagnose those causes. Exclusions match case-insensitive substrings, so `SK3d` removes all matching directions while `SK1h_Yp` selects a narrower subset.

The curve data come from header totals, not the summary printed in cell 7. This exclusion list affects this plotting call only.

In [20]:
excluded_sats_for_delta_v_plot = [
    'SK3d',      # These display noise issues (100s of m vibrations in 0.001s)
    'SK1h_Yp',   # Manoeuver optimisation did not apply correctly, uses same dv for each maneuver and diverges
    'SK1h_Ym',   # Manoeuver optimisation did not apply correctly, uses same dv for each maneuver and diverges
    'SK1h_Zp',   # Manoeuver optimisation did not apply correctly, uses same dv for each maneuver and diverges
    'SK1h_Zm',   # Manoeuver optimisation did not apply correctly, uses same dv for each maneuver and diverges
    'SK0p5h_Zp', # not simulated yet
    'SK0p5h_Zm', # not simulated yet
    'SK0p5h_Yp', # Manoeuver optimisation did not apply correctly, uses same dv for each maneuver and diverges
    'SK0p5h_Ym', # not simulated yet
    ]
plot_delta_v_vs_sk_duration(
                            exclude_names=excluded_sats_for_delta_v_plot,
                            y_log=True,
                            x_axis_days=True,
                            )


Plot saved to: c:\Users\megap\Desktop\LIFE\Working\LIFE-Sem-Project\Previous-Project\automation\results\delta_v_vs_sk_duration_xdays_ylog.png


## 20. Calculate deviations and populate satellite_data

This cell calls `calculate_max_deviations`, writes the deviation CSVs, prints each successfully processed case's maxima in metres, and stores the grouped results in `satellite_data` for later plots.

The scan includes all matching position CSVs, including cases later excluded from comparison plots. If you change the source position data, rerun this cell to refresh both the saved deviation tables and the in-memory results before regenerating figures.

In [21]:
satellite_data = calculate_max_deviations()



CALCULATING POSITION DEVIATIONS

  ✓ Octahedron_SK28d_Xm1000m
      Max deviations - X: 255.635m, Y: 111.832m, Z: 102.960m, Mag: 279.028m
  ✓ Octahedron_SK28d_Xp1000m
      Max deviations - X: 255.549m, Y: 112.619m, Z: 103.085m, Mag: 278.818m
  ✓ Octahedron_SK28d_Ym1000m
      Max deviations - X: 113.199m, Y: 153.882m, Z: 24.302m, Mag: 159.949m
  ✓ Octahedron_SK28d_Yp1000m
      Max deviations - X: 112.575m, Y: 154.941m, Z: 24.315m, Mag: 161.019m
  ✓ Octahedron_SK28d_Zm1000m
      Max deviations - X: 104.744m, Y: 23.535m, Z: 176.937m, Mag: 205.942m
  ✓ Octahedron_SK28d_Zp1000m
      Max deviations - X: 104.735m, Y: 23.533m, Z: 177.052m, Mag: 205.804m


## 21. Generate the four-panel maximum-deviation figure

This cell defines the separate `excluded_sats` list used for deviation comparisons and trade-off figures. Besides the original problematic or unavailable cases, it excludes `SK1d_Xm` and `SK1d_Zm` as annotated in the source.

It calls `plot_max_deviations` with logarithmic axes. The four panels show maximum absolute x/y/z deviations and maximum vector magnitude for the retained cases. Filtering affects the plot; it does not remove cases from `satellite_data` or delete their deviation CSVs.

In [22]:
excluded_sats = [
    'SK3d',      # These display noise issues (100s of m vibrations in 0.001s)
    'SK1d_Xm',   # Unknown issue causing oscillation at 170 days
    'SK1d_Zm',   # Unknown issue causing oscillation at 170 days
    'SK1h_Yp',   # Manoeuver optimisation did not apply correctly, uses same dv for each maneuver and diverges
    'SK1h_Ym',   # Manoeuver optimisation did not apply correctly, uses same dv for each maneuver and diverges
    'SK1h_Zp',   # Manoeuver optimisation did not apply correctly, uses same dv for each maneuver and diverges
    'SK1h_Zm',   # Manoeuver optimisation did not apply correctly, uses same dv for each maneuver and diverges
    'SK0p5h_Zp', # not simulated yet
    'SK0p5h_Zm', # not simulated yet
    'SK0p5h_Yp', # Manoeuver optimisation did not apply correctly, uses same dv for each maneuver and diverges
    'SK0p5h_Ym', # not simulated yet
    ]
plot_max_deviations(satellite_data,
                    x_log=True,
                    y_log=True,
                    exclude_names = excluded_sats,
                    )


Plot saved to: c:\Users\megap\Desktop\LIFE\Working\LIFE-Sem-Project\Previous-Project\automation\results\position_deviations_vs_sk_xlog_ylog.png


## 22. Generate the magnitude-only comparison figure

This cell calls `plot_magnitude_deviations` using the same `satellite_data` and `excluded_sats` list, with logarithmic axes.

It creates a separate saved figure containing the magnitude statistic already present in the fourth panel of the preceding comparison. It does not recalculate deviations or regenerate their CSVs.

In [23]:
plot_magnitude_deviations(satellite_data,
                         x_log=True,
                         y_log=True,
                         exclude_names=excluded_sats)


Plot saved to: c:\Users\megap\Desktop\LIFE\Working\LIFE-Sem-Project\Previous-Project\automation\results\magnitude_deviation_vs_sk_xlog_ylog.png


## 23. Inspect the Ym satellite at the one-hour interval

This cell plots the time history for `Octahedron_SK1h_Ym1000m`, using days on the time axis. The commented alternative for `SK6h` is retained but does not execute.

This direct inspection still includes the case even though `SK1h_Ym` appears in the comparison exclusion lists. It reads the deviation CSV produced in cell 20, saves the four-panel time-series figure and displays it. If that case was unavailable or failed processing, the function prints its missing path and returns.

In [24]:
plot_deviations_over_time(
                          "Ym1000m", # Example satellite
                          sk_interval="SK1h",
                        #   "Ym1000m",
                        #   sk_interval="SK6h",
                          x_axis_days=True,
                          )


✗ File not found: c:\Users\megap\Desktop\LIFE\Working\LIFE-Sem-Project\Previous-Project\automation\data-export\deviations\Octahedron_SK1h_Ym1000m_deviations.csv

Available files in c:\Users\megap\Desktop\LIFE\Working\LIFE-Sem-Project\Previous-Project\automation\data-export\deviations:
  - Octahedron_SK28d_Xm1000m_deviations.csv
  - Octahedron_SK28d_Xp1000m_deviations.csv
  - Octahedron_SK28d_Ym1000m_deviations.csv
  - Octahedron_SK28d_Yp1000m_deviations.csv
  - Octahedron_SK28d_Zm1000m_deviations.csv
  - Octahedron_SK28d_Zp1000m_deviations.csv


## 24. Inspect Xp time histories for eight SK intervals

This loop plots `Xp1000m` for `SK3h`, `SK6h`, `SK12h`, `SK1d`, `SK3d`, `SK7d`, `SK14d` and `SK28d`. Each available case generates its own four-panel time-series PNG and plot window. The half-hour and one-hour entries remain commented out.

All calls use the days option and therefore assume 10-minute sampling. The loop includes `SK3d` for inspection even though that interval is excluded from the comparison figures. Missing cases are reported and skipped.

In [25]:
for interval in [
    # 'SK0p5h',
    # 'SK1h',
    'SK3h',
    'SK6h',
    'SK12h',
    'SK1d',
    'SK3d',
    'SK7d',
    'SK14d',
    'SK28d',
    ]:
    plot_deviations_over_time(
                              "Xp1000m", # Example satellite
                              sk_interval=interval,
                              x_axis_days=True,
                              )


✗ File not found: c:\Users\megap\Desktop\LIFE\Working\LIFE-Sem-Project\Previous-Project\automation\data-export\deviations\Octahedron_SK3h_Xp1000m_deviations.csv

Available files in c:\Users\megap\Desktop\LIFE\Working\LIFE-Sem-Project\Previous-Project\automation\data-export\deviations:
  - Octahedron_SK28d_Xm1000m_deviations.csv
  - Octahedron_SK28d_Xp1000m_deviations.csv
  - Octahedron_SK28d_Ym1000m_deviations.csv
  - Octahedron_SK28d_Yp1000m_deviations.csv
  - Octahedron_SK28d_Zm1000m_deviations.csv
  - Octahedron_SK28d_Zp1000m_deviations.csv
✗ File not found: c:\Users\megap\Desktop\LIFE\Working\LIFE-Sem-Project\Previous-Project\automation\data-export\deviations\Octahedron_SK6h_Xp1000m_deviations.csv

Available files in c:\Users\megap\Desktop\LIFE\Working\LIFE-Sem-Project\Previous-Project\automation\data-export\deviations:
  - Octahedron_SK28d_Xm1000m_deviations.csv
  - Octahedron_SK28d_Xp1000m_deviations.csv
  - Octahedron_SK28d_Ym1000m_deviations.csv
  - Octahedron_SK28d_Yp1000m_dev

## 25. Generate the trade-off figure with logarithmic axes

This cell plots mean Δv against the mean of individual maximum magnitude deviations for each SK interval, using logarithmic x and y scales and `excluded_sats`.

Only cases with both deviation data and a matching readable maneuver-header total contribute. The resulting PNG has `_xlog_ylog` and exclusion suffixes. It is a cross-satellite average comparison; inspect the individual-satellite plots when assessing the most demanding spacecraft.

In [26]:
plot_delta_v_vs_max_deviation(satellite_data,
                             x_log=True,
                             y_log=True,
                             exclude_names=excluded_sats)


Plot saved to: c:\Users\megap\Desktop\LIFE\Working\LIFE-Sem-Project\Previous-Project\automation\results\delta_v_vs_max_deviation_xlog_ylog_exclude_sk3d_sk1d_xm_sk1d_zm_sk1h_yp_sk1h_ym_sk1h_zp_sk1h_zm_sk0p5h_zp_sk0p5h_zm_sk0p5h_yp_sk0p5h_ym.png


## 26. Generate the trade-off figure with a linear Δv axis

This final code cell repeats the same trade-off calculation and exclusions, keeping the deviation axis logarithmic and changing the Δv axis to linear.

It saves a separate PNG with `_xlog` and the exclusion suffix, allowing comparison with the previous view. The underlying matching and averaging procedure is unchanged. This completes the source script's active execution sequence.

In [27]:
plot_delta_v_vs_max_deviation(satellite_data,
                             x_log=True,
                             y_log=False,
                             exclude_names=excluded_sats)


Plot saved to: c:\Users\megap\Desktop\LIFE\Working\LIFE-Sem-Project\Previous-Project\automation\results\delta_v_vs_max_deviation_xlog_exclude_sk3d_sk1d_xm_sk1d_zm_sk1h_yp_sk1h_ym_sk1h_zp_sk1h_zm_sk0p5h_zp_sk0p5h_zm_sk0p5h_yp_sk0p5h_ym.png


## Official Ansys STK references and result checks

- [Maneuver Summary](https://help.agi.com/stk/SubSystems/dataProviders/Content/html/dataProviders/Maneuver_Summary.htm): per-maneuver Δv definitions for Astrogator reports.
- [Vector Choose Axes](https://help.agi.com/stkdevkit/12.6.0/Subsystems/dataProviders/Content/html/dataProviders/Vector_Choose_Axes.htm): coordinate components depend on the selected reference axes.
- [Generating Reports and Graphs](https://help.agi.com/stk/Content/stk/report.htm): report export and selection of time intervals and sampling.

The documentation explains STK quantities; the CSV metadata marker, naming conventions, grouping, exclusions and averaging are specific to this script.

Before interpreting the results, check that the input reports cover comparable durations, use the intended coordinate convention, and have 600-second sampling for the time-series day axis. Check the printed Δv comparisons and any skipped-file messages. Each plotting function saves its PNG before displaying it; the printed path identifies the output. No summary tables are automatically exported by this script.

The supplied attachment contains code only. The notebook has been checked for Python syntax and preservation of the source code, but its numerical output has not been verified against your exported datasets.
